# NYC taxi pickups with Uber H3-style hexagons

This notebook demonstrates hexagonal aggregation without requiring a Python H3 wheel. The map uses browser-side JavaScript and h3-js from a CDN, which is often more reliable in JupyterLite/Pyodide than native Python geospatial wheels.

Data: a tiny browser-training sample of NYC taxi pickup points. For a larger open dataset, use NYC TLC trip records or Kaggle mirrors when credentials are available.

Read/watch prompt: Uber introduced H3 as a hierarchical hexagonal indexing system for city-scale spatial analysis; use this notebook to compare points versus cells.

**Reflection questions:** What detail is lost when points become hexagons? Why might hexagons reduce visual bias compared with square grids? What resolution would you choose for neighborhood versus citywide analysis?

In [ ]:
# Pyodide/JupyterLite bootstrap: install only pure-Python packages used in this notebook.
import sys, importlib
try:
    import micropip
except Exception:
    micropip = None

async def ensure_packages(packages):
    for pkg, import_name in packages:
        try:
            importlib.import_module(import_name)
        except Exception:
            if micropip is None:
                raise RuntimeError(f'{pkg} is not installed and micropip is unavailable.')
            await micropip.install(pkg)

await ensure_packages([('pandas','pandas'), ('folium','folium'), ('branca','branca'), ('plotly','plotly')])


In [ ]:
from pathlib import Path
import json, math, statistics
import pandas as pd
import folium
from folium.plugins import MarkerCluster, HeatMap, TimestampedGeoJson, MiniMap, Fullscreen, MeasureControl

DATA = Path('../data')

def load_json(name):
    return json.loads((DATA / name).read_text(encoding='utf-8'))

def load_csv(name):
    return pd.read_csv(DATA / name)

def add_standard_controls(m):
    MiniMap(toggle_display=True).add_to(m)
    Fullscreen().add_to(m)
    MeasureControl(primary_length_unit='kilometers').add_to(m)
    folium.LayerControl(collapsed=False).add_to(m)
    return m

def color_scale(values, colors=('green','orange','red')):
    vals = list(values)
    lo, hi = min(vals), max(vals)
    def pick(v):
        if hi == lo:
            return colors[1]
        t = (v - lo) / (hi - lo)
        return colors[0] if t < .33 else colors[1] if t < .66 else colors[2]
    return pick


In [ ]:
taxi = load_csv('nyc_taxi_pickups_tiny.csv')
taxi

In [ ]:
# A pure HTML/JavaScript H3 map. This works in the browser and keeps Python dependency-light.
from IPython.display import HTML
points = taxi.to_dict(orient='records')
html_doc = f'''
<div id="h3map" style="height: 620px; width: 100%;"></div>
<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css"/>
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
<script src="https://unpkg.com/h3-js@4.1.0/dist/h3-js.umd.js"></script>
<script>
const pts = {json.dumps(points)};
const map = L.map('h3map').setView([40.74, -73.96], 11);
L.tileLayer('https://tile.openstreetmap.org/{{z}}/{{x}}/{{y}}.png', {{maxZoom: 19, attribution: 'OpenStreetMap'}}).addTo(map);
const counts = {{}};
pts.forEach(p => {{
  const cell = h3.latLngToCell(p.lat, p.lon, 8);
  counts[cell] = (counts[cell] || 0) + 1;
  L.circleMarker([p.lat, p.lon], {{radius: 4}}).bindPopup(`${{p.pickup_zone}}<br>${{p.pickup_datetime}}`).addTo(map);
}});
Object.entries(counts).forEach(([cell, count]) => {{
  const boundary = h3.cellToBoundary(cell).map(x => [x[0], x[1]]);
  L.polygon(boundary, {{weight: 1, fillOpacity: Math.min(0.15 + count*0.18, 0.85)}})
    .bindPopup(`H3 cell: ${{cell}}<br>Pickups: ${{count}}`)
    .addTo(map);
}});
</script>
'''
HTML(html_doc)